# Titanic Survival Prediction - Feature Engineering Task
### ML Internship - Hecta AI Solution

Dataset: **Titanic - Machine Learning from Disaster (Kaggle)**

This notebook covers:
1. Loading data from Google Drive
2. Data cleaning (missing values)
3. Feature engineering (new features from existing columns)
4. Feature selection (correlation + feature importance)
5. Model training (baseline vs main model)
6. Evaluation and comparison


## Step 1: Mount Google Drive and Load Dataset

The Kaggle Titanic dataset downloads as a zip file containing three files:

- `train.csv` - has the `Survived` column, this is what we use to train and evaluate
- `test.csv` - no `Survived` column, used only for Kaggle submission
- `gender_submission.csv` - an example submission format, not needed here

We only need `train.csv` for this task, since it's the only file with labels.
Upload the zip file (e.g. `titanic.zip`) to your Google Drive, then unzip it directly
inside Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# Change this path to where you uploaded the zip file in your Google Drive
zip_path = '/content/drive/MyDrive/titanic.zip'
extract_path = '/content/titanic_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

import os
os.listdir(extract_path)


In [ ]:
df = pd.read_csv(extract_path + '/train.csv')
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


## Step 2: Data Cleaning

- `Age` has missing values -> fill with median age
- `Embarked` has missing values -> fill with the most common port
- `Cabin` has too many missing values -> we will extract the Deck letter, then drop the original column
- `Fare` (if missing in test data) -> fill with median fare


In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())

df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df['Fare'] = df['Fare'].fillna(df['Fare'].median())

df.isnull().sum()


## Step 3: Feature Engineering

We create new features that help the model understand the data better:

- **Title**: extracted from the passenger's Name (Mr, Mrs, Miss, Master, Rare)
- **FamilySize**: SibSp + Parch + 1 (the passenger themselves)
- **IsAlone**: 1 if the passenger has no family aboard, else 0
- **Deck**: first letter of the Cabin column (missing cabins become 'U' for Unknown)
- **AgeBin**: Age grouped into ranges (Child, Teen, Adult, Senior)
- **FareBin**: Fare split into 4 equal-sized groups (quartiles)


In [ ]:
df['Title'] = df['Name'].str.extract(r',\s*([^.]*)\.')

df['Title'] = df['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major',
     'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

df['Title'].value_counts()


In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['IsAlone'] = 0
df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

df[['FamilySize', 'IsAlone']].head()


In [ ]:
df['Deck'] = df['Cabin'].str[0]
df['Deck'] = df['Deck'].fillna('U')

df['Deck'].value_counts()


In [ ]:
df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 60, 100],
                      labels=['Child', 'Teen', 'Adult', 'Senior'])

df['FareBin'] = pd.qcut(df['Fare'], 4, labels=['Low', 'Mid', 'High', 'VeryHigh'])

df[['AgeBin', 'FareBin']].head()


In [ ]:
# Drop columns we no longer need
df = df.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'])

df.head()


## Step 4: Encoding Categorical Features

Machine learning models need numbers, not text categories.
We use `pd.get_dummies` (one-hot encoding) for all categorical columns.


In [ ]:
categorical_cols = ['Sex', 'Embarked', 'Title', 'Deck', 'AgeBin', 'FareBin']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df_encoded.head()


## Step 5: Feature Selection

We check two things:
1. **Correlation with Survived** - which features move together with the target
2. **Feature importance from Random Forest** - which features the model actually relies on


In [ ]:
correlation = df_encoded.corr()['Survived'].sort_values(ascending=False)
correlation


In [ ]:
plt.figure(figsize=(8, 10))
sns.barplot(x=correlation.values[1:], y=correlation.index[1:])
plt.title('Correlation with Survived')
plt.xlabel('Correlation')
plt.show()


In [ ]:
X = df_encoded.drop(columns=['Survived'])
y = df_encoded['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, X_test.shape


In [ ]:
rf_for_importance = RandomForestClassifier(n_estimators=200, random_state=42)
rf_for_importance.fit(X_train, y_train)

importances = pd.Series(rf_for_importance.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(8, 10))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.show()


## Step 6: Baseline Model

A simple Logistic Regression is our baseline. Any "real" model should beat this.


In [ ]:
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_preds)

print('Baseline (Logistic Regression) Accuracy:', baseline_accuracy)
print(classification_report(y_test, baseline_preds))


## Step 7: Main Model (Random Forest)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_preds)

print('Random Forest Accuracy:', rf_accuracy)
print(classification_report(y_test, rf_preds))


In [ ]:
cm = confusion_matrix(y_test, rf_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
plt.title('Random Forest - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## Step 8: Compare Baseline vs Main Model

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression (Baseline)', 'Random Forest'],
    'Accuracy': [baseline_accuracy, rf_accuracy]
})

comparison


## Conclusion

- Engineered features (Title, FamilySize, IsAlone, Deck, AgeBin, FareBin) give the model
  more useful signal than the raw columns alone.
- The Random Forest model is compared against a simple Logistic Regression baseline
  to confirm the extra complexity is actually earning its keep.
- See the accompanying Word document (`Titanic_Code_Explanation.docx`) for a
  plain-language walkthrough of every step above.
